# 03 — Sales by Product Category
Sales distribution and absolute revenue/profit across Furniture, Office Supplies, and Technology.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Sales Distribution by Category ───────────────────────────────────────────
cat = fetch_df(cur, """
    SELECT p.category,
           ROUND(SUM(oi.sales),  2) AS sales,
           ROUND(SUM(oi.profit), 2) AS profit
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY sales DESC
""")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sales Distribution by Product Category",
             fontsize=14, fontweight="bold", y=1.02)

# Pie chart – sales share
wedges, texts, autotexts = axes[0].pie(
    cat["sales"], labels=cat["category"],
    autopct="%1.1f%%", colors=PALETTE[:len(cat)],
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
axes[0].set_title("Sales Share")

# Bar chart – absolute sales & profit side by side
bar_w = 0.35
x     = range(len(cat))
b1 = axes[1].bar([i - bar_w/2 for i in x], cat["sales"],
                 bar_w, color=PALETTE[0], label="Sales",  edgecolor="white")
b2 = axes[1].bar([i + bar_w/2 for i in x], cat["profit"],
                 bar_w, color=PALETTE[1], label="Profit", edgecolor="white")

for bar in b1:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 500,
                 fmt_usd(bar.get_height()),
                 ha="center", va="bottom", fontsize=8, color=PALETTE[0])
for bar in b2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 200,
                 fmt_usd(bar.get_height()),
                 ha="center", va="bottom", fontsize=8, color=PALETTE[1])

axes[1].set_xticks(list(x))
axes[1].set_xticklabels(cat["category"], fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
axes[1].legend(frameon=False)
axes[1].set_title("Sales & Profit (USD)")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
